## 1. Imports and Configuration

In [1]:
import tidy3d.web as web  # if needed
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from cycler import cycler
from scipy.optimize import curve_fit
from scipy.signal import hilbert

import tidy3d as td
from tidy3d.plugins.resonance import ResonanceFinder
import gdstk


print(f"tidy3d version: {td.__version__}")
web.test()

tidy3d version: 2.11.2


14:09:05 W. Europe Daylight Time Authentication configured successfully!

In [13]:
class build_nanobeam_cavity:
    # default values are for the SiV cavity
    def __init__(
        self,
        period=0.260,
        hole_radius=0.075,
        wg_width=0.365,
        ang=25,
        n_hole=20,
        n_taper=8,
        end_wg_length=5.0,
        ellipse_tolerance=0.001,
    ):
        # Parameters
        # ----------
        # period        : mirror-hole lattice period [um]
        # hole_radius   : ideal circular hole radius [um]
        # wg_width      : nanobeam full width [um]
        # ang           : triangle half-angle / sidewall-related angle [deg]
        # n_hole        : number of holes on each side
        # n_taper       : number of taper holes on each side
        # end_wg_length : straight waveguide length added beyond the holes on each end [um]

        self.period = float(period)
        self.hole_radius = float(hole_radius)
        self.wg_width = float(wg_width)
        self.ang = float(ang)
        self.n_hole = int(n_hole)
        self.n_taper = int(n_taper)
        self.end_wg_length = float(end_wg_length)
        self.ellipse_tolerance = float(ellipse_tolerance)

        if self.n_taper < 2:
            raise ValueError("n_taper must be at least 2.")
        if self.n_hole < self.n_taper:
            raise ValueError("n_hole must be >= n_taper.")
        if self.end_wg_length < 0:
            raise ValueError("end_wg_length must be non-negative.")

        self.thickness = self.wg_width / (2 * np.tan(np.deg2rad(self.ang)))

        self._precompute_cavity_params()
        self._precompute_geometry()

    def _precompute_cavity_params(self):
        i = np.arange(self.n_taper, dtype=float)
        frac = i / (self.n_taper - 1)

        # Kalish taper:
        # a_i = 0.9a + 0.1a * (i / (N - 1))^2

        self.a_list = 0.9 * self.period + 0.1 * self.period * frac**2
        # self.a_list: [a0, a1, ..., a_{n_taper-1}]

        self.a_cumsum = np.cumsum(self.a_list)
        # self.a_cumsum:  [a0, a0+a1, a0+a1+a2, ...]
        self.a_total = float(self.a_cumsum[-1])

        self.n_mirror_eff = self.n_hole - self.n_taper

    def _precompute_geometry(self):
        # Positive-x taper hole centers.
        # Equivalent to:
        # positions[0] = a_taper[0] / 2
        # positions[i] = positions[i - 1] + a_taper[i]
        taper_positions = (
            self.a_list[0] / 2
            + np.r_[
                0.0,
                np.cumsum(self.a_list[1:]),
            ]
        )

        # Positive-x mirror hole centers after taper.
        mirror_positions = taper_positions[-1] + self.period * np.arange(
            1,
            self.n_mirror_eff + 1,
            dtype=float,
        )

        positive_positions = np.r_[taper_positions, mirror_positions]

        # Full symmetric hole array.
        self.HOLE_X_POSITIONS_UM = np.r_[
            -positive_positions[::-1],
            positive_positions,
        ]

        # Perfect circular holes: no fabrication error.
        self.HOLE_RADIUS_X_UM = np.full_like(
            self.HOLE_X_POSITIONS_UM,
            self.hole_radius,
            dtype=float,
        )
        self.HOLE_CENTER_Y_UM = 0.0
        self.HOLE_RADIUS_Y_UM = self.hole_radius

        # GDS ellipse polygon approximation tolerance, not fabrication error.
        self.ELLIPSE_TOLERANCE_UM = self.ellipse_tolerance

        self.positive_hole_positions_um = positive_positions
        self.outer_hole_x_um = float(positive_positions[-1])

        # Beam extends past the last hole by half a period plus straight wg.
        self.beam_half_length_um = (
            self.outer_hole_x_um + self.period / 2 + self.end_wg_length
        )
        self.cav_len = 2 * self.beam_half_length_um

    def get_hole_specs(self):
        return {
            "HOLE_X_POSITIONS_UM": self.HOLE_X_POSITIONS_UM,
            "HOLE_RADIUS_X_UM": self.HOLE_RADIUS_X_UM,
            "HOLE_CENTER_Y_UM": self.HOLE_CENTER_Y_UM,
            "HOLE_RADIUS_Y_UM": self.HOLE_RADIUS_Y_UM,
            "ELLIPSE_TOLERANCE_UM": self.ELLIPSE_TOLERANCE_UM,
        }


print(build_nanobeam_cavity().get_hole_specs()["HOLE_X_POSITIONS_UM"])

[-4.94928571 -4.68928571 -4.42928571 -4.16928571 -3.90928571 -3.64928571
 -3.38928571 -3.12928571 -2.86928571 -2.60928571 -2.34928571 -2.08928571
 -1.82928571 -1.56928571 -1.31618367 -1.06891837 -0.82642857 -0.58765306
 -0.35153061 -0.117       0.117       0.35153061  0.58765306  0.82642857
  1.06891837  1.31618367  1.56928571  1.82928571  2.08928571  2.34928571
  2.60928571  2.86928571  3.12928571  3.38928571  3.64928571  3.90928571
  4.16928571  4.42928571  4.68928571  4.94928571]


In [12]:
# ── Physical parameters ───────────────────────────────────────────────────────
WAVELENGTH_SCOUT = 0.737  # Broadband centre wavelength [µm]  (737 nm)
SIDEWALL_ANGLE_DEG = 25 # Fabrication sidewall angle [degrees]
# NA = 0.65  # Collection numerical aperture
N_BG = 1.0  # Background refractive index (air)
C0 = 299_792_458.0  # Speed of light [m/s]

############################################################################
PERIOD_UM = 0.260

SiV_Cavity = build_nanobeam_cavity(
    period=PERIOD_UM,
    hole_radius=0.075,
    wg_width=0.365,
    ang=SIDEWALL_ANGLE_DEG,
    n_hole=20,
    n_taper=8,
    end_wg_length=4.0,
    ellipse_tolerance=0.001,
)

HOLE_X_POSITIONS_UM = SiV_Cavity.get_hole_specs()["HOLE_X_POSITIONS_UM"]
HOLE_RADIUS_X_UM = SiV_Cavity.get_hole_specs()["HOLE_RADIUS_X_UM"]
HOLE_CENTER_Y_UM = SiV_Cavity.get_hole_specs()["HOLE_CENTER_Y_UM"]
HOLE_RADIUS_Y_UM = SiV_Cavity.get_hole_specs()["HOLE_RADIUS_Y_UM"]
ELLIPSE_TOLERANCE_UM = SiV_Cavity.get_hole_specs()["ELLIPSE_TOLERANCE_UM"]


CAVITY_BBOX_UM = (
    -SiV_Cavity.cav_len / 2,
    -SiV_Cavity.wg_width / 2,
    SiV_Cavity.cav_len / 2,
    SiV_Cavity.wg_width / 2,
)


##########################################################################


# ── Notebook-local runtime paths and embedded geometry ──────────────────────
# This notebook can generate its own GDS files from explicit cavity and hole
# specs, so no external geometry files are required.
RUNTIME_ROOT = Path.cwd().resolve() / "tidy3d_SiV_diamond_photonic_cavity_runtime"
GDS_DIR = RUNTIME_ROOT / "gds"
RESULTS_DIR = RUNTIME_ROOT / "data" / "results"
GDS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def generate_local_gds_from_specs(
    gds_dir: Path,
    cavity_bbox_um: tuple[float, float, float, float],
    hole_x_positions_um: np.ndarray,
    hole_radius_x_um: np.ndarray,
    hole_center_y_um: float,
    hole_radius_y_um: float,
    ellipse_tolerance_um: float = 0.001,
    force: bool = False,
    subtract_holes: bool = True,
) -> tuple[Path, Path]:
    cavity_path = gds_dir / "Cavity_Ideal.gds"
    holes_path = gds_dir / "Holes_Ideal.gds"

    if cavity_path.exists() and holes_path.exists() and not force:
        return cavity_path, holes_path

    holes = [
        gdstk.ellipse(
            (float(x_um), float(hole_center_y_um)),
            (float(rx_um), float(hole_radius_y_um)),
            tolerance=float(ellipse_tolerance_um),
            layer=0,
            datatype=0,
        )
        for x_um, rx_um in zip(hole_x_positions_um, hole_radius_x_um)
    ]

    holes_lib = gdstk.Library(unit=1e-6, precision=1e-9)
    holes_top = gdstk.Cell("TOP")
    for hole in holes:
        holes_top.add(hole)
    holes_lib.add(holes_top)
    holes_lib.write_gds(str(holes_path))

    xmin, ymin, xmax, ymax = cavity_bbox_um
    rect = gdstk.rectangle(
        (float(xmin), float(ymin)),
        (float(xmax), float(ymax)),
        layer=0,
        datatype=0,
    )

    cavity_polys = gdstk.boolean([rect], holes, "not") if subtract_holes else [rect]

    cavity_lib = gdstk.Library(unit=1e-6, precision=1e-9)
    cavity_top = gdstk.Cell("TOP")
    for poly in cavity_polys or [rect]:
        cavity_top.add(poly)
    cavity_lib.add(cavity_top)
    cavity_lib.write_gds(str(cavity_path))

    return cavity_path, holes_path


CAVITY_GDS, HOLES_GDS = generate_local_gds_from_specs(
    gds_dir=GDS_DIR,
    cavity_bbox_um=CAVITY_BBOX_UM,
    hole_x_positions_um=HOLE_X_POSITIONS_UM,
    hole_radius_x_um=HOLE_RADIUS_X_UM,
    hole_center_y_um=HOLE_CENTER_Y_UM,
    hole_radius_y_um=HOLE_RADIUS_Y_UM,
    ellipse_tolerance_um=ELLIPSE_TOLERANCE_UM,
    force=True,
    subtract_holes=True,
)

In [14]:
# ── Publication-style plot theme ──────────────────────────────────────────────
RED = "#840032"
BLUE = "#002642"
YELLOW = "#e59500"
WHITE = "#FFFAF2"
BLACK = "#02040f"
PALETTE = [RED, BLUE, YELLOW, BLACK]

# Monopolar colormap: black → blue → red → yellow → white
mono_cmap = LinearSegmentedColormap.from_list(
    "mono", [BLACK, BLUE, RED, YELLOW, WHITE], N=256
)
# Bipolar colormap: black → blue → white → yellow → red
bipolar_cmap = LinearSegmentedColormap.from_list(
    "bipolar", [BLACK, BLUE, WHITE, YELLOW, RED], N=256
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["axes.prop_cycle"] = cycler(color=PALETTE)
plt.rcParams["figure.autolayout"] = True
plt.rcParams["font.size"] = 11

print("Plot theme ready.")

Plot theme ready.


## 2. Define the Diamond Material Model

Diamond's refractive index is described by a two-term **Sellmeier equation** (Zaitsev 2001):

$$n^2(\lambda) = 1 + \frac{B_1 \lambda^2}{\lambda^2 - C_1} + \frac{B_2 \lambda^2}{\lambda^2 - C_2}$$

with $B_1 = 0.3306$, $C_1 = 0.175^2\,\mu\text{m}^2$, $B_2 = 4.3356$, $C_2 = 0.106^2\,\mu\text{m}^2$. The model is valid from 0.23 to 5 µm.

In [ ]:
def n_diamond(wavelength_um):
    """
    Diamond refractive index via the two-term Sellmeier equation.

    Reference: Zaitsev, Optical Properties of Diamond (2001).
    Valid range: 0.23 – 5 µm.

    Parameters
    ----------
    wavelength_um : float  — free-space wavelength [µm]

    Returns
    -------
    float  — refractive index at the given wavelength
    """
    lam2 = np.asarray(wavelength_um, dtype=float) ** 2
    B1, C1 = 0.3306, 0.175**2  # first oscillator
    B2, C2 = 4.3356, 0.106**2  # second oscillator
    n2 = 1.0 + B1 * lam2 / (lam2 - C1) + B2 * lam2 / (lam2 - C2)
    return np.sqrt(n2)


# Tidy3D dispersive medium (Sellmeier coefficients)
diamond_medium = td.Sellmeier(coeffs=[(0.3306, 0.175**2), (4.3356, 0.106**2)])
air_medium = td.Medium(permittivity=1.0)

n_scout = float(n_diamond(SCOUT_WAVELENGTH_UM))
print(f"n_diamond at {SCOUT_WAVELENGTH_UM * 1e3:.0f} nm : {n_scout:.4f}")

# ── Dispersion curve ──────────────────────────────────────────────────────────
wl = np.linspace(0.45, 1.0, 400)
n_wl = n_diamond(wl)

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(wl * 1e3, n_wl, color=RED, lw=2, label="Diamond")
ax.axvline(
    SCOUT_WAVELENGTH_UM * 1e3,
    color=BLUE,
    ls="--",
    lw=1.5,
    label=f"{SCOUT_WAVELENGTH_UM * 1e3:.0f} nm  (n = {n_scout:.3f})",
)
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Refractive index $n$")
ax.set_title("Diamond Dispersion (Sellmeier)")
ax.legend(framealpha=0.85)
ax.set_xlim(450, 1000)
plt.tight_layout()
plt.show()